# Práctica 2. ChromaDB paso a paso

Ya habéis visto un vector en la práctica 1. Ahora ese vector entra en una base de datos.

ChromaDB es la base del proyecto DocIA+. Aquí trabaja en memoria, dentro de Colab. Al final veréis una colección que sí se escribe en disco.

Cada registro guarda cuatro cosas:

- `ids`: identificador único
- `documents`: el texto que se podría citar
- `embeddings`: la lista de números, calculada por nosotros
- `metadatas`: la categoría, para filtrar

Si solo pasáis el texto y no el vector, ChromaDB llama a un modelo que no veis. En este cuaderno no se hace así.

Las frases siguen siendo de ejercicio.


## Instalar y cargar el mismo modelo


In [ ]:
!pip install chromadb sentence-transformers -q

from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("all-MiniLM-L6-v2")
print("Modelo listo. Dimensiones:", modelo.get_sentence_embedding_dimension())


## Crear la colección

En SQL esto sería una tabla. Aquí se llama colección. El espacio es el coseno: más adelante veréis por qué. Hoy basta con saber que una distancia más pequeña significa más parecido.


In [ ]:
import chromadb

cliente = chromadb.EphemeralClient()
existentes = [c.name for c in cliente.list_collections()]
if "documentacion_ies" in existentes:
    cliente.delete_collection("documentacion_ies")

coleccion = cliente.create_collection(
    name="documentacion_ies",
    metadata={"hnsw:space": "cosine"},
)
print("Coleccion creada:", coleccion.name)


## Guardar cinco frases

Primero se calcula el vector. Después se guarda, junto con el texto y la categoría.


In [ ]:
frases = [
    "Procedimiento de formalizacion de matricula en el curso de especializacion.",
    "Para matricularse hay que presentar la solicitud en secretaria.",
    "El horario de la cafeteria del centro es de 8:00 a 14:00.",
    "Las pruebas de acceso a ciclos formativos se realizan en junio.",
    "El plan de orientacion ayuda al alumnado a elegir estudios.",
]
categorias = [
    {"categoria": "Oferta Educativa", "fuente": "ejemplo-oferta"},
    {"categoria": "Secretaria", "fuente": "ejemplo-acceso"},
    {"categoria": "Servicios", "fuente": "ejemplo-horarios"},
    {"categoria": "Secretaria", "fuente": "ejemplo-acceso"},
    {"categoria": "Orientacion", "fuente": "ejemplo-orientacion"},
]
ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]
vectores = modelo.encode(frases).tolist()

coleccion.add(
    ids=ids,
    documents=frases,
    embeddings=vectores,
    metadatas=categorias,
)
print("Registros guardados:", coleccion.count())


## Mirar el vector que quedó dentro

Tiene que salir 384, los mismos que en la práctica 1.


In [ ]:
interno = coleccion.get(ids=["doc1"], include=["embeddings", "documents"])
texto = interno["documents"][0]
vector = interno["embeddings"][0]
print("Texto:", texto)
print("Dimensiones:", len(vector))
print("Diez primeros numeros:", list(vector[:10]))


## Preguntar con otras palabras

La pregunta no contiene «formalizacion» ni «matricula». El vector de la pregunta se calcula con el mismo modelo y se compara con los que están guardados.


In [ ]:
pregunta = "Como me inscribo en el curso?"
vector_pregunta = modelo.encode([pregunta]).tolist()
resultados = coleccion.query(query_embeddings=vector_pregunta, n_results=2)

print("Pregunta:", pregunta)
print()
for i, doc in enumerate(resultados["documents"][0]):
    distancia = resultados["distances"][0][i]
    categoria = resultados["metadatas"][0][i]["categoria"]
    print(f"{i + 1}. distancia={distancia:.4f}  categoria={categoria}")
    print(f"   {doc}")
    print()


## Filtrar por categoría

La cercanía y el filtro no son la misma cosa. Aquí la pregunta es amplia, pero solo pueden salir textos de `Secretaria`.


In [ ]:
pregunta = "Cuando hay que hacer los tramites?"
vector_pregunta = modelo.encode([pregunta]).tolist()
filtrados = coleccion.query(
    query_embeddings=vector_pregunta,
    n_results=2,
    where={"categoria": "Secretaria"},
)
print("Pregunta:", pregunta)
print("Filtro: categoria = Secretaria")
print()
for doc, meta in zip(filtrados["documents"][0], filtrados["metadatas"][0]):
    print("-", doc, f"({meta['categoria']})")


## Memoria y disco

La colección de arriba desaparece al cerrar el cuaderno. Una base de datos también tiene que seguir ahí mañana. `PersistentClient` escribe una carpeta.


In [ ]:
from pathlib import Path

ruta = Path("/content/chroma_ies")
cliente_disco = chromadb.PersistentClient(path=str(ruta))
if "documentacion_ies" in [c.name for c in cliente_disco.list_collections()]:
    cliente_disco.delete_collection("documentacion_ies")

en_disco = cliente_disco.create_collection(
    name="documentacion_ies",
    metadata={"hnsw:space": "cosine"},
)
en_disco.add(ids=ids, documents=frases, embeddings=vectores, metadatas=categorias)

otro = chromadb.PersistentClient(path=str(ruta))
abierta = otro.get_collection("documentacion_ies")
print("Registros al volver a abrir la carpeta:", abierta.count())
print("Carpeta:", ruta)


## Ejercicio

1. Añade el texto «El centro dispone de aparcamiento para bicicletas.» con el id `doc6` y la categoría `Servicios`.
2. Calcula tú el vector con `modelo.encode`. No dejes que ChromaDB lo invente.
3. Pregunta «Donde puedo dejar la bicicleta?» y pide un solo resultado.
4. Imprime el texto y la distancia.

La distancia tiene que ser pequeña, y la categoría tiene que ser `Servicios`.


In [ ]:
# 1. Calcula el vector de la frase nueva.
# 2. coleccion.add(...) o en_disco.add(...)
# 3. Pregunta y muestra la distancia.
